In [1]:
!pip install langchain langchain-community langchain-core langchain-text-splitters langchain-anthropic sentence-transformers faiss-cpu pypdf --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


##Imports

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_anthropic import ChatAnthropic

/tmp/ipykernel_10254/2263093767.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Split PDF

In [12]:
pdf = PyPDFLoader("/content/Supply Chain PDF2.pdf")
document = pdf.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
Chunk = splitter.split_documents(document)

print(f"PDF loaded and split into {len(Chunk)} chunks")

PDF loaded and split into 49 chunks


## Embeddings and Vector Store

In [13]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(documents=Chunk, embedding=embeddings)
retriever = db.as_retriever(search_kwargs={"k": 3})

print("Vector store ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store ready


## Load Anthropic

In [11]:
from google.colab import userdata
ANTHROPIC_KEY = "API Key"
llm = ChatAnthropic(
    model="claude-haiku-4-5",
    api_key=ANTHROPIC_KEY
)
print("LLM ready")

LLM ready


## RAG Chain with memory

In [14]:
chat_history = []

def ask(question):
    # Step 1: retrieve relevant chunks from PDF
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Step 2: build last 4 exchanges as history
    history_text = ""
    for role, msg in chat_history[-4:]:
        history_text += f"{role}: {msg}\n"

    # Step 3: build prompt
    full_prompt = f"""You are a helpful and friendly assistant who answers questions about a document.
Be conversational and natural — if someone greets you or chats casually, respond warmly.
For document questions, use the context provided.
If you don't know the answer from the document, say so honestly.

Previous conversation:
{history_text}

Relevant document context:
{context}

User: {question}
Assistant:"""

    # Step 4: call LLM
    result = llm.invoke(full_prompt)
    answer = result.content

    # Step 5: save to history
    chat_history.append(("User", question))
    chat_history.append(("Assistant", answer))

    return answer

print("RAG chain ready")

RAG chain ready


##Interactive Chat

In [10]:
print("💬 Chat with your PDF! Type 'quit' to exit or 'reset' to clear memory.\n")

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue
    elif user_input.lower() == "quit":
        print("Goodbye!")
        break
    elif user_input.lower() == "reset":
        chat_history = []
        print("🔄 Memory cleared!\n")
        continue

    response = ask(user_input)
    print(f"\nAssistant: {response}\n")

💬 Chat with your PDF! Type 'quit' to exit or 'reset' to clear memory.

You: what is this document about?

Assistant: Hi! This document is about **supply chain management** and its importance to companies. 

Based on the table of contents, it covers things like:

- The characteristics of supply chains (including shared information, organizational relationships, inventory management, and coordination)
- Major reasons why companies use supply chains (like reducing inventory investment)
- Transportation and shipment requirements
- The role of third-party logistics providers

It looks like a research report that was sponsored by the Department of Transportation's University Transportation Centers Program. So it's a pretty comprehensive overview of how supply chains work and why they matter for business operations.

Is there anything specific about supply chain management you'd like to know more about?

You: what are successful strategic alliance/partnership has the following characteristics